In [3]:
# One-page PDF export (key features)
import textwrap
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

# If key_features_df doesn't exist, build it from Feature_Details.json
if 'key_features_df' not in globals():
    # Prefer already-loaded variables if present, otherwise load from disk
    if 'variables' not in globals():
        # Try common locations
        feature_json_path = Path('coursework') / 'Feature_Details.json'
        if not feature_json_path.exists():
            feature_json_path = Path('Feature_Details.json')
        with open(feature_json_path, 'r', encoding='utf-8') as f:
            variables = json.load(f)

    key_features = {
        'Target Variables': ['hospdead', 'death'],
        'Demographics': ['age', 'sex', 'race'],
        'Vital Signs': ['meanbp', 'hrt', 'resp', 'temp'],
        'Lab Results': ['glucose', 'crea', 'wblc', 'ph'],
        'Disease Classification': ['dzgroup', 'dzclass', 'num.co']
    }

    rows = []
    for section, var_list in key_features.items():
        for var in var_list:
            info = variables.get(var)
            if not info:
                rows.append({
                    'Section': section,
                    'Variable': var,
                    'Type': '',
                    'Missing Values': '',
                    'Meaning (Health Context)': 'Not found in Feature_Details.json',
                    'Clinical Significance': ''
                })
                continue
            rows.append({
                'Section': section,
                'Variable': var,
                'Type': info.get('type', ''),
                'Missing Values': info.get('missing_values', ''),
                'Meaning (Health Context)': info.get('description', ''),
                'Clinical Significance': info.get('clinical_significance', '')
            })

    key_features_df = pd.DataFrame(rows)

pdf_path = Path('key_features_table_one_page.pdf')
df_pdf = key_features_df.copy()

# Wrap long text so it fits on one page
wrap_cols = ['Meaning (Health Context)', 'Clinical Significance']
for c in wrap_cols:
    if c in df_pdf.columns:
        df_pdf[c] = df_pdf[c].fillna('').astype(str).apply(
            lambda s: "\n".join(textwrap.wrap(s, width=60))
        )

# Limit to key columns for readability
cols = ['Section', 'Variable', 'Type', 'Meaning (Health Context)', 'Clinical Significance']
cols = [c for c in cols if c in df_pdf.columns]
df_pdf = df_pdf[cols]

# Create a single-page figure (A4 landscape-ish)
fig_w, fig_h = 11.69, 8.27  # inches ~ A4 landscape
fig, ax = plt.subplots(figsize=(fig_w, fig_h))
ax.axis('off')
ax.set_title('SUPPORT2 — Key Features (Clinical Meanings)', fontsize=14, fontweight='bold', pad=12)

table = ax.table(
    cellText=df_pdf.values,
    colLabels=df_pdf.columns,
    cellLoc='left',
    colLoc='left',
    loc='upper center'
 )

table.auto_set_font_size(False)
table.set_fontsize(7)
table.scale(1, 1.35)

# Make header readable
for (row, col), cell in table.get_celld().items():
    if row == 0:
        cell.set_text_props(fontweight='bold')
        cell.set_facecolor('#e6e6e6')

plt.tight_layout()
plt.savefig(pdf_path, dpi=300, bbox_inches='tight')
plt.close(fig)

print(f"Saved one-page PDF: {pdf_path.resolve()}")

Saved one-page PDF: D:\year-2\smartDataDiscovery\coursework\key_features_table_one_page.pdf
